In [35]:
from dataclasses import dataclass
import os
from typing import Optional
from openai import OpenAI
import json
from pydantic import BaseModel
@dataclass(frozen=True)
class Provider:
    """One provider to reliably route requests across all inference providers"""
    name:str
    env_var:str
    is_free:bool
    base_url: Optional[str]
    model:str

PROVIDERS = [
   Provider("Groq","GROQ_API_KEY",True,"https://api.groq.com/openai/v1","openai/gpt-oss-120b"),
   Provider("OpenAI","OPENAI_API_KEY",True,None,"gpt-4o-mini"),    
]

def select_provider() -> Provider:
    for provider in PROVIDERS:
        if os.getenv(provider.env_var):
            return provider

    expected = ", ".join(p.env_var for p in PROVIDERS)
    raise RuntimeError(f"No provider key set, add one of {expected} to your environment variables")

def build_client(provider: Provider) -> OpenAI:


    api_key = os.getenv(provider.env_var)
    if provider.base_url is None:
        return OpenAI(api_key=api_key)

    return OpenAI(
        api_key=api_key,
        base_url=provider.base_url
    )


def have_any_key()->bool:
    return any(os.getenv(p.env_var) for p in PROVIDERS)

print("Found a provider key." if have_any_key() else "No provider key found.")    


def llm_reply(prompt:str)->str:
    provider = select_provider()
    print(provider)
    client = build_client(provider)
    result = client.chat.completions.create(
        model = provider.model,
        max_tokens=200,
        messages=[
            {
                "role":"user",
                "content":prompt,
            }
        ]
    )

    return result.choices[0].message.content


Found a provider key.


In [3]:
ZERO_SHOT_COT_SUFFIX = "\n\nLets think step by step." 

def zero_shot_cot_prompt(question:str)->str:
    """This function takes a question and returns a prompt for the zero shot COT model."""
    return f"Question:{question}{ZERO_SHOT_COT_SUFFIX}"

In [5]:
QUESTION = (
    "Roger has 5 tennis balls. He buys 2 more cans of tennis balls."
    "Each can has 3 tennis balls. How many tennis balls does he have now?"
)

In [6]:
print(zero_shot_cot_prompt(QUESTION))

Question:Roger has 5 tennis balls. He buys 2 more cans of tennis balls.Each can has 3 tennis balls. How many tennis balls does he have now?

Lets think step by step.


In [7]:
reply = llm_reply(zero_shot_cot_prompt(QUESTION))
print(reply)

Provider(name='Groq', env_var='GROQ_API_KEY', is_free=True, base_url='https://api.groq.com/openai/v1', model='openai/gpt-oss-120b')
**Step‑by‑step solution**

1. **Start with the balls Roger already has**  
   - He has **5** tennis balls.

2. **Find out how many balls are in the new cans**  
   - He buys **2** cans.  
   - Each can contains **3** tennis balls.  
   - Total balls from the cans = \(2 \times 3 = 6\).

3. **Add the new balls to the ones he already has**  
   - Balls he already has: 5  
   - Balls from the cans: 6  
   - Total now = \(5 + 6 = 11\).




In [42]:
SYSTEM_PROMPT = """
You are a product-review sentiment analyst.
Read each review the user sends and classify its sentiment as positive, negative, or neutral.
We should also give a confidence score for the sentiment in range 0-1.

For each review, respond with only a json object - no other text or comments - in strictly the following shape.


{
    "sentiment": "positive" | "negative" | "neutral",
    "confidence": 0.0-1.0,  
    "reason": "short explanation for the sentiment",
}

""".strip()

def analyze_review(incoming_review:str)->str:
    """Send review to LLM and ask it to classify the sentiment."""
    provider = select_provider()
    client = build_client(provider)

    result = client.chat.completions.create(
        model = provider.model,
        max_tokens = 100,
        messages = [
            {
                "role":"system",
                "content":SYSTEM_PROMPT,
            },
            {
                "role":"user",
                "content":incoming_review,

            }
        ],

    )

    return result.choices[0].message.content.strip()



In [ ]:
from this import d
from typing import Literal


REVIEWS= [
    "I love this product! It's amazing!",
    "This is the worst product I've ever used.",
    "I'm not sure I can recommend this to anyone.",
]

review_analysis = analyze_review(REVIEWS[0])
print(review_analysis)
    

class SentimentResult(BaseModel):
    """
    Sentiment result from the LLM.
    """

    sentiment: Literal["positive","negative","neutral"]
    confidence: float
    reason: str

def validate_sentiment_reply(raw_reply:str)-> SentimentResult | str:
    try:
        response = json.loads(raw_reply)
        return SentimentResult(**response)
    except json.JSONDecodeError:
        return f"Invalid JSON:{raw_reply}"
    
result = validate_sentiment_reply(review_analysis)
print(result)
    
def route_by_sentiment(result:SentimentResult)->str:
    """
    Route the review based on the sentiment.
    """
    if result.sentiment == "positive":
        return "Positive review"
    elif result.sentiment == "negative":
        return "Negative review, connect with customer service"
    else:
        return "Neutral review, no action needed"

route_by_sentiment(result)





{
    "sentiment": "positive",
    "confidence": 0.99,
    "reason": "Enthusiastic language and positive adjectives ('love', 'amazing') indicate a clear positive sentiment."
}
sentiment='positive' confidence=0.99 reason="Enthusiastic language and positive adjectives ('love', 'amazing') indicate a clear positive sentiment."


'Positive review'